## Building Attention Modules

Welcome to the fourth and final lesson of "Sequence Models & The Dawn of Attention"! We've accomplished so much together in this course. We started by understanding the limitations of RNNs and LSTMs, then discovered how attention mechanisms revolutionize sequence modeling through the Query-Key-Value paradigm. In our most recent lesson, we mastered scaled dot-product attention and learned how masking enables real-world applications by handling padding tokens and maintaining causal ordering for autoregressive generation.

Today, we're taking a significant step toward building production-ready attention systems by creating a standalone PyTorch module for scaled dot-product attention. Rather than writing attention as a simple function, we'll encapsulate it in a reusable `nn.Module` that can be easily integrated into larger architectures. This modular approach mirrors how real Transformer implementations work, where attention is a fundamental building block that is reused across multiple layers. We'll design our module to handle various input configurations, support masking, and include essential features like dropout for robust training.

---

### The Need for Modular Attention

As we move toward building complex architectures like Transformers, treating attention as a standalone function becomes limiting. Real neural networks require components that can maintain state, participate in gradient computation graphs, and integrate seamlessly with PyTorch's training infrastructure. This is where `nn.Module` shines: it provides automatic parameter registration, gradient tracking, and clean interfaces for complex operations.

Our modular attention design will accept pre-projected Query, Key, and Value tensors as inputs. This separation of concerns is crucial because, in full Transformer architectures, the same attention module is reused across different layers and heads, while the projection matrices (that create Q, K, V from input embeddings) vary. By focusing our module purely on the attention computation itself, we create a flexible building block that can serve multiple purposes.

Additionally, a proper module allows us to incorporate training-specific features like dropout, which is essential for preventing overfitting in large attention-based models. The module will also handle the complex broadcasting requirements for masks, making it robust enough for various input configurations we'll encounter in real applications.

---

### Designing the `ScaledDotProductAttention` Class

Let's begin building our attention module by establishing the class structure and constructor. Our module needs to inherit from `nn.Module` and handle the essential components for attention computation:

```python
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class ScaledDotProductAttention(nn.Module):
    def __init__(self, dropout=0.1):
        """
        Scaled Dot-Product Attention module

        Args:
            dropout: Dropout rate for attention weights
        """
        super(ScaledDotProductAttention, self).__init__()
        self.dropout = nn.Dropout(dropout)
```

Our constructor is deliberately simple, focusing on the essential parameters needed for attention computation. The `dropout` parameter controls regularization applied to attention weights, which is a standard technique in Transformer training to prevent overfitting. We initialize an `nn.Dropout` layer that we'll apply to attention weights before computing the final output.

Notice that we don't store any learned parameters in this module; the attention computation itself is parameter-free. The actual learned parameters (projection matrices for creating Q, K, V) will live in higher-level modules that use our attention as a building block. This design keeps our attention module focused and reusable.

---

### Implementing the Forward Pass

Now let's implement the complete forward pass that computes scaled dot-product attention. This method encapsulates all the mathematical operations we've studied, including attention score computation, masking, softmax normalization, and dropout:

```python
def forward(self, query, key, value, mask=None):
    """
    Forward pass for scaled dot-product attention

    Args:
        query: (batch_size (B), num_heads (H), seq_len_q (L_q), d_k) or (batch_size, seq_len_q, d_k)
        key: (batch_size, num_heads, seq_len_k (L_k), d_k) or (batch_size, seq_len_k, d_k)
        value: (batch_size, num_heads, seq_len_v (L_v), d_v) or (batch_size, seq_len_v, d_v)
        mask: (batch_size, 1, seq_len_q, seq_len_k) or (batch_size, seq_len_q, seq_len_k)
            Mask should be broadcastable to the attention score shape.

    Returns:
        output: (batch_size, num_heads, seq_len_q, d_v) or (batch_size, seq_len_q, d_v)
        attention_weights: (batch_size, num_heads, seq_len_q, seq_len_k) or (batch_size, seq_len_q, seq_len_k)
    """
    d_k = query.size(-1)

    # Compute attention scores with scaling
    # scores shape: (batch_size [B], num_heads [H], seq_len_q [L_q], seq_len_k [L_k])
    scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)

    # Apply mask if provided
    if mask is not None:
        # Mask shape should be broadcastable to scores shape
        # (batch_size [B], 1, 1, seq_len_k [L_k]) for padding mask on keys
        # (batch_size [B], 1, seq_len_q [L_q], seq_len_k [L_k]) for combined padding/look-ahead
        # (1, 1, seq_len_q [L_q], seq_len_k [L_k]) for look-ahead mask
        # Masked positions (where mask == 0) are set to a large negative value so their softmax becomes ~0
        scores = scores.masked_fill(mask == 0, -1e9)

    # Apply softmax to get attention weights
    # attention_weights shape: (batch_size [B], num_heads [H], seq_len_q [L_q], seq_len_k [L_k])
    attention_weights = F.softmax(scores, dim=-1)
    attention_weights = self.dropout(attention_weights)  # Apply dropout to attention weights

    # Compute output by multiplying attention weights with values
    # output shape: (batch_size [B], num_heads [H], seq_len_q [L_q], d_v)
    output = torch.matmul(attention_weights, value)

    return output, attention_weights
```

The `forward` method signature is designed for maximum flexibility. Our tensors can have either 3 dimensions for single-head attention or 4 dimensions for multi-head scenarios. The `transpose(-2, -1)` operation swaps the last two dimensions of the key tensor, which works correctly regardless of tensor dimensionality. The scaling factor `math.sqrt(d_k)` ensures numerical stability by preventing the dot products from growing too large, as we learned in the previous lesson.

The masking logic uses `masked_fill` to set masked positions to very large negative values (`-1e9`), which become essentially zero after softmax. This approach is more numerically stable than setting them to negative infinity. The detailed comments about mask shapes help users understand the various broadcasting patterns they might encounter in practice.

Applying dropout to attention weights (rather than the final output) is a subtle but important design choice from the original Transformer paper. This technique randomly zeroes some attention connections during training, forcing the model to maintain diverse attention strategies and preventing overfitting to specific patterns.

---

### Creating Test Utilities

Before testing our module, let's create utility functions that generate realistic test data and demonstrate our module's flexibility:

```python
def create_sample_qkv(batch_size, seq_len_q, seq_len_k, d_k, d_v):
    """Create sample Q, K, V tensors"""
    torch.manual_seed(42)
    q = torch.randn(batch_size, seq_len_q, d_k, requires_grad=True)
    k = torch.randn(batch_size, seq_len_k, d_k, requires_grad=True)
    v = torch.randn(batch_size, seq_len_k, d_v, requires_grad=True)  # seq_len_k == seq_len_v
    return q, k, v
```

This utility function creates sample Query, Key, and Value tensors with different sequence lengths and dimensions. The `requires_grad=True` parameter ensures our tensors participate in gradient computation, validating that our module integrates properly with PyTorch's automatic differentiation system.

Notice how the value tensor shares its sequence length with the key tensor (`seq_len_k`). This is not a coincidence: in attention mechanisms, the set of positions over which we attend (the keys) must exactly match the set of positions from which we gather information (the values). This means that for every key position, there is a corresponding value position, and the attention weights computed for each query are used to take a weighted sum over these value positions. As a result, the sequence length of the value tensor (`seq_len_v`) must always be equal to the sequence length of the key tensor (`seq_len_k`). This alignment is a fundamental property of all attention mechanisms.

---

### Comprehensive Module Testing

Now let's create comprehensive tests to validate that our attention module works correctly across different scenarios. We'll test both masked and unmasked attention to ensure our implementation handles various real-world conditions:

```python
def main():
    print("Testing ScaledDotProductAttention Module")

    # Configuration
    batch_size = 2
    seq_len_q = 5
    seq_len_k = 7  # seq_len_v will be same as seq_len_k
    d_k = 16  # Dimension of keys/queries
    d_v = 20  # Dimension of values

    # Create sample data
    query, key, value = create_sample_qkv(batch_size, seq_len_q, seq_len_k, d_k, d_v)

    print(f"Input shapes - Q: {query.shape}, K: {key.shape}, V: {value.shape}")

    # Initialize attention module
    attention_module = ScaledDotProductAttention(dropout=0.1)

    # Test forward pass without mask
    output_no_mask, attn_weights_no_mask = attention_module(query, key, value)
    print(f"Output shape (no mask): {output_no_mask.shape}")
    print(f"Attention weights shape (no mask): {attn_weights_no_mask.shape}")
    assert output_no_mask.shape == (batch_size, seq_len_q, d_v)
    assert attn_weights_no_mask.shape == (batch_size, seq_len_q, seq_len_k)

    # Test with a mask
    mask = torch.ones(batch_size, seq_len_q, seq_len_k)
    mask[:, :, -2:] = 0  # Mask out last two key positions for all queries

    output_masked, attn_weights_masked = attention_module(query, key, value, mask=mask)
    print(f"Output shape (masked): {output_masked.shape}")
    print(f"Attention weights shape (masked): {attn_weights_masked.shape}")
    assert output_masked.shape == (batch_size, seq_len_q, d_v)
```

Our test setup creates realistic scenarios where queries and keys have different sequence lengths (5 vs. 7), simulating cross-attention scenarios common in encoder-decoder architectures. We use different dimensions for keys/queries (`d_k=16`) and values (`d_v=20`) to test that our module correctly handles scenarios where the output dimension differs from the input dimension. This flexibility is essential in real Transformer architectures, where different components may use different dimensional representations.

The masking test creates a simple scenario where we block attention to the last two key positions. This simulates a common real-world case where certain positions (like future tokens or padding) should be ignored. The assertions ensure our module produces outputs with the expected shapes, confirming dimensional correctness.

When we run our complete test, it produces the following output:

```text
Testing ScaledDotProductAttention Module
Input shapes - Q: torch.Size([2, 5, 16]), K: torch.Size([2, 7, 16]), V: torch.Size([2, 7, 20])
Output shape (no mask): torch.Size([2, 5, 20])
Attention weights shape (no mask): torch.Size([2, 5, 7])
Output shape (masked): torch.Size([2, 5, 20])
Attention weights shape (masked): torch.Size([2, 5, 7])
```

This output confirms our module correctly transforms the input dimensions: we start with queries of shape `(2, 5, 16)` and values of shape `(2, 7, 20)`, and produce outputs of shape `(2, 5, 20)`. The attention mechanism properly aggregates information from all 7 key-value positions to produce representations for each of the 5 query positions, with the final dimension matching the value dimension.

---

### Conclusion and Next Steps

Congratulations! You've reached the end of our course "Sequence Models & The Dawn of Attention", the first course in the "Exploring the Transformer Architecture" course path! What an incredible journey we've taken together. We started by understanding the fundamental limitations of RNNs and LSTMs, discovered how attention mechanisms revolutionize sequence modeling, mastered scaled dot-product attention with comprehensive masking strategies, and today built a production-ready attention module that serves as a cornerstone of modern NLP architectures. The `ScaledDotProductAttention` module we've created isn't just an academic exercise: it's the exact type of building block used in real Transformer implementations.

Your journey into the world of Transformers is just beginning! In our next course, "Deconstructing the Transformer Architecture", we'll take these attention fundamentals and build the complete Transformer architecture from the ground up. We'll explore multi-head attention, positional encoding, layer normalization, and the encoder-decoder structure that powers models like GPT and BERT. Get ready to see how all these pieces come together to create the most influential architecture in modern AI!

## Building Your First Attention Module

After learning about scaled dot-product attention theory, you're ready to build your first reusable attention module! Your objective is to complete the initialization of the `ScaledDotProductAttention` class so that it's ready to perform attention calculations.

---

### Your Task

Looking at the provided code, you'll notice that the `__init__` method has the basic structure set up, but it's missing a crucial component. The `forward` method already expects to use a dropout layer on the attention weights, so you need to make sure this layer is properly created during initialization.

Your job is to add the missing line that creates the dropout layer using the dropout rate passed as a parameter. Once you complete this, the module will be ready to handle attention computations with proper regularization!

---

### Starter Code

```python
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class ScaledDotProductAttention(nn.Module):
    def __init__(self, dropout=0.1):
        """
        Scaled Dot-Product Attention module
        
        Args:
            dropout: Dropout rate
        """
        super(ScaledDotProductAttention, self).__init__()
        # TODO: Create dropout layer
        
    def forward(self, query, key, value, mask=None):
        """
        Forward pass for scaled dot-product attention
        
        Args:
            query: (batch_size, num_heads, seq_len_q, d_k) or (batch_size, seq_len_q, d_k)
            key: (batch_size, num_heads, seq_len_k, d_k) or (batch_size, seq_len_k, d_k)
            value: (batch_size, num_heads, seq_len_v, d_v) or (batch_size, seq_len_v, d_v)
                   (seq_len_k and seq_len_v must be the same)
            mask: (batch_size, 1, seq_len_q, seq_len_k) or (batch_size, seq_len_q, seq_len_k)
                  Mask should be broadcastable.
                  
        Returns:
            output: (batch_size, num_heads, seq_len_q, d_v) or (batch_size, seq_len_q, d_v)
            attention_weights: (batch_size, num_heads, seq_len_q, seq_len_k) or (batch_size, seq_len_q, seq_len_k)
        """
        d_k = query.size(-1)
        
        # Compute attention scores with scaling: (B, [H], L_q, L_k)
        scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)
        
        # Apply mask if provided
        if mask is not None:
            # Ensure mask is broadcastable. Common mask shapes:
            # (B, 1, 1, L_k) for padding mask on keys
            # (B, 1, L_q, L_k) for combined padding/look-ahead
            # (1, 1, L_q, L_k) for look-ahead mask
            scores = scores.masked_fill(mask == 0, -1e9)
        
        # Apply softmax to get attention weights: (B, [H], L_q, L_k)
        attention_weights = F.softmax(scores, dim=-1)
        attention_weights = self.dropout(attention_weights) # Apply dropout to attention weights
        
        # Compute output by multiplying attention weights with values: (B, [H], L_q, d_v)
        output = torch.matmul(attention_weights, value)
        
        return output, attention_weights

def create_sample_qkv(batch_size, seq_len_q, seq_len_k, d_k, d_v):
    """Create sample Q, K, V tensors"""
    torch.manual_seed(42)
    q = torch.randn(batch_size, seq_len_q, d_k, requires_grad=True)
    k = torch.randn(batch_size, seq_len_k, d_k, requires_grad=True)
    v = torch.randn(batch_size, seq_len_k, d_v, requires_grad=True) # seq_len_k == seq_len_v
    return q, k, v
    
def main():
    print("Testing ScaledDotProductAttention Module")
    
    # Configuration
    batch_size = 2
    seq_len_q = 5
    seq_len_k = 7 # seq_len_v will be same as seq_len_k
    d_k = 16 # Dimension of keys/queries
    d_v = 20 # Dimension of values
    
    # Create sample data
    query, key, value = create_sample_qkv(batch_size, seq_len_q, seq_len_k, d_k, d_v)
    
    print(f"Input shapes - Q: {query.shape}, K: {key.shape}, V: {value.shape}")
    
    # Initialize attention module
    attention_module = ScaledDotProductAttention(dropout=0.1)
    
    # Test forward pass without mask
    output_no_mask, attn_weights_no_mask = attention_module(query, key, value)
    print(f"Output shape (no mask): {output_no_mask.shape}")
    print(f"Attention weights shape (no mask): {attn_weights_no_mask.shape}")
    assert output_no_mask.shape == (batch_size, seq_len_q, d_v)
    assert attn_weights_no_mask.shape == (batch_size, seq_len_q, seq_len_k)
    
    # Test with a mask
    mask = torch.ones(batch_size, seq_len_q, seq_len_k)
    mask[:, :, -2:] = 0 # Mask out last two key positions for all queries
    
    output_masked, attn_weights_masked = attention_module(query, key, value, mask=mask)
    print(f"Output shape (masked): {output_masked.shape}")
    print(f"Attention weights shape (masked): {attn_weights_masked.shape}")
    assert output_masked.shape == (batch_size, seq_len_q, d_v)
    
    # Verify masking worked: attention weights should be 0 for masked positions
    masked_positions = attn_weights_masked[:, :, -2:] # Last 2 key positions
    print(f"Masked positions attention (should be 0): {masked_positions.max():.6f}")
    

if __name__ == "__main__":
    main()
```

---

### Solution

Only the dropout layer in `__init__` was missing:

```python
self.dropout = nn.Dropout(dropout)
```

- `nn.Dropout(dropout)` creates a dropout layer using the rate passed to the constructor (`0.1` by default).
- Storing it as `self.dropout` registers it as a submodule, so `forward` can call `self.dropout(attention_weights)`, and PyTorch switches it on in `train()` mode and off in `eval()` mode automatically.
- Without this line, `forward` raises `AttributeError: 'ScaledDotProductAttention' object has no attribute 'dropout'`.
- The module still has no learned parameters: dropout only randomly zeroes some attention weights during training to regularize the model.

Running the test shows the expected shapes for both the unmasked and masked cases: queries `(2, 5, 16)` and values `(2, 7, 20)` produce an output of `(2, 5, 20)` and attention weights of `(2, 5, 7)`. The final check prints `0.000000` for the two masked key positions, confirming they receive no attention.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class ScaledDotProductAttention(nn.Module):
    def __init__(self, dropout=0.1):
        """
        Scaled Dot-Product Attention module
        
        Args:
            dropout: Dropout rate
        """
        super(ScaledDotProductAttention, self).__init__()
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, query, key, value, mask=None):
        """
        Forward pass for scaled dot-product attention
        
        Args:
            query: (batch_size, num_heads, seq_len_q, d_k) or (batch_size, seq_len_q, d_k)
            key: (batch_size, num_heads, seq_len_k, d_k) or (batch_size, seq_len_k, d_k)
            value: (batch_size, num_heads, seq_len_v, d_v) or (batch_size, seq_len_v, d_v)
                   (seq_len_k and seq_len_v must be the same)
            mask: (batch_size, 1, seq_len_q, seq_len_k) or (batch_size, seq_len_q, seq_len_k)
                  Mask should be broadcastable.
                  
        Returns:
            output: (batch_size, num_heads, seq_len_q, d_v) or (batch_size, seq_len_q, d_v)
            attention_weights: (batch_size, num_heads, seq_len_q, seq_len_k) or (batch_size, seq_len_q, seq_len_k)
        """
        d_k = query.size(-1)
        
        # Compute attention scores with scaling: (B, [H], L_q, L_k)
        scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)
        
        # Apply mask if provided
        if mask is not None:
            # Ensure mask is broadcastable. Common mask shapes:
            # (B, 1, 1, L_k) for padding mask on keys
            # (B, 1, L_q, L_k) for combined padding/look-ahead
            # (1, 1, L_q, L_k) for look-ahead mask
            scores = scores.masked_fill(mask == 0, -1e9)
        
        # Apply softmax to get attention weights: (B, [H], L_q, L_k)
        attention_weights = F.softmax(scores, dim=-1)
        attention_weights = self.dropout(attention_weights) # Apply dropout to attention weights
        
        # Compute output by multiplying attention weights with values: (B, [H], L_q, d_v)
        output = torch.matmul(attention_weights, value)
        
        return output, attention_weights

def create_sample_qkv(batch_size, seq_len_q, seq_len_k, d_k, d_v):
    """Create sample Q, K, V tensors"""
    torch.manual_seed(42)
    q = torch.randn(batch_size, seq_len_q, d_k, requires_grad=True)
    k = torch.randn(batch_size, seq_len_k, d_k, requires_grad=True)
    v = torch.randn(batch_size, seq_len_k, d_v, requires_grad=True) # seq_len_k == seq_len_v
    return q, k, v
    
def main():
    print("Testing ScaledDotProductAttention Module")
    
    # Configuration
    batch_size = 2
    seq_len_q = 5
    seq_len_k = 7 # seq_len_v will be same as seq_len_k
    d_k = 16 # Dimension of keys/queries
    d_v = 20 # Dimension of values
    
    # Create sample data
    query, key, value = create_sample_qkv(batch_size, seq_len_q, seq_len_k, d_k, d_v)
    
    print(f"Input shapes - Q: {query.shape}, K: {key.shape}, V: {value.shape}")
    
    # Initialize attention module
    attention_module = ScaledDotProductAttention(dropout=0.1)
    
    # Test forward pass without mask
    output_no_mask, attn_weights_no_mask = attention_module(query, key, value)
    print(f"Output shape (no mask): {output_no_mask.shape}")
    print(f"Attention weights shape (no mask): {attn_weights_no_mask.shape}")
    assert output_no_mask.shape == (batch_size, seq_len_q, d_v)
    assert attn_weights_no_mask.shape == (batch_size, seq_len_q, seq_len_k)
    
    # Test with a mask
    mask = torch.ones(batch_size, seq_len_q, seq_len_k)
    mask[:, :, -2:] = 0 # Mask out last two key positions for all queries
    
    output_masked, attn_weights_masked = attention_module(query, key, value, mask=mask)
    print(f"Output shape (masked): {output_masked.shape}")
    print(f"Attention weights shape (masked): {attn_weights_masked.shape}")
    assert output_masked.shape == (batch_size, seq_len_q, d_v)
    
    # Verify masking worked: attention weights should be 0 for masked positions
    masked_positions = attn_weights_masked[:, :, -2:] # Last 2 key positions
    print(f"Masked positions attention (should be 0): {masked_positions.max():.6f}")
    

if __name__ == "__main__":
    main()


## Building the Attention Core

Great work on setting up the dropout layer in your attention module! Now it's time to implement the core calculation that makes attention work.

---

### Your Task

In this exercise, you need to complete the forward pass by implementing the scaled attention score calculation. The provided code has everything set up, but the crucial line for computing the attention scores is missing. You'll need to perform matrix multiplication between the query and key tensors, and then apply the proper scaling factor.

Remember that scaled dot-product attention follows the formula:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

Your job is to implement the first part of this calculation: computing the dot product of the query and the transposed key, then applying the scaling factor. The rest of the forward pass (masking, softmax, dropout, and final output) is already implemented and will work correctly once you provide the attention scores. This step is fundamental to understanding how attention mechanisms work!

---

### Starter Code

```python
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class ScaledDotProductAttention(nn.Module):
    def __init__(self, dropout=0.1):
        """
        Scaled Dot-Product Attention module
        
        Args:
            dropout: Dropout rate
        """
        super(ScaledDotProductAttention, self).__init__()
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, query, key, value, mask=None):
        """
        Forward pass for scaled dot-product attention
        
        Args:
            query: (batch_size, num_heads, seq_len_q, d_k) or (batch_size, seq_len_q, d_k)
            key: (batch_size, num_heads, seq_len_k, d_k) or (batch_size, seq_len_k, d_k)
            value: (batch_size, num_heads, seq_len_v, d_v) or (batch_size, seq_len_v, d_v)
                   (seq_len_k and seq_len_v must be the same)
            mask: (batch_size, 1, seq_len_q, seq_len_k) or (batch_size, seq_len_q, seq_len_k)
                  Mask should be broadcastable.
                  
        Returns:
            output: (batch_size, num_heads, seq_len_q, d_v) or (batch_size, seq_len_q, d_v)
            attention_weights: (batch_size, num_heads, seq_len_q, seq_len_k) or (batch_size, seq_len_q, seq_len_k)
        """
        d_k = query.size(-1)
        
        # Compute attention scores with scaling: (B, [H], L_q, L_k)
        # TODO: Implement scaled attention score calculation
        
        # Apply mask if provided
        if mask is not None:
            # Ensure mask is broadcastable. Common mask shapes:
            # (B, 1, 1, L_k) for padding mask on keys
            # (B, 1, L_q, L_k) for combined padding/look-ahead
            # (1, 1, L_q, L_k) for look-ahead mask
            scores = scores.masked_fill(mask == 0, -1e9)
        
        # Apply softmax to get attention weights: (B, [H], L_q, L_k)
        attention_weights = F.softmax(scores, dim=-1)
        attention_weights = self.dropout(attention_weights) # Apply dropout to attention weights
        
        # Compute output by multiplying attention weights with values: (B, [H], L_q, d_v)
        output = torch.matmul(attention_weights, value)
        
        return output, attention_weights

def create_sample_qkv(batch_size, seq_len_q, seq_len_k, d_k, d_v):
    """Create sample Q, K, V tensors"""
    torch.manual_seed(42)
    q = torch.randn(batch_size, seq_len_q, d_k, requires_grad=True)
    k = torch.randn(batch_size, seq_len_k, d_k, requires_grad=True)
    v = torch.randn(batch_size, seq_len_k, d_v, requires_grad=True) # seq_len_k == seq_len_v
    return q, k, v
    
def main():
    print("Testing ScaledDotProductAttention Module")
    
    # Configuration
    batch_size = 2
    seq_len_q = 5
    seq_len_k = 7 # seq_len_v will be same as seq_len_k
    d_k = 16 # Dimension of keys/queries
    d_v = 20 # Dimension of values
    
    # Create sample data
    query, key, value = create_sample_qkv(batch_size, seq_len_q, seq_len_k, d_k, d_v)
    
    print(f"Input shapes - Q: {query.shape}, K: {key.shape}, V: {value.shape}")
    
    # Initialize attention module
    attention_module = ScaledDotProductAttention(dropout=0.1)
    
    # Test forward pass without mask
    output_no_mask, attn_weights_no_mask = attention_module(query, key, value)
    print(f"Output shape (no mask): {output_no_mask.shape}")
    print(f"Attention weights shape (no mask): {attn_weights_no_mask.shape}")
    assert output_no_mask.shape == (batch_size, seq_len_q, d_v)
    assert attn_weights_no_mask.shape == (batch_size, seq_len_q, seq_len_k)
    
    # Test with a mask
    mask = torch.ones(batch_size, seq_len_q, seq_len_k)
    mask[:, :, -2:] = 0 # Mask out last two key positions for all queries
    
    output_masked, attn_weights_masked = attention_module(query, key, value, mask=mask)
    print(f"Output shape (masked): {output_masked.shape}")
    print(f"Attention weights shape (masked): {attn_weights_masked.shape}")
    assert output_masked.shape == (batch_size, seq_len_q, d_v)
    
    # Verify masking worked: attention weights should be 0 for masked positions
    masked_positions = attn_weights_masked[:, :, -2:] # Last 2 key positions
    print(f"Masked positions attention (should be 0): {masked_positions.max():.6f}")
    

if __name__ == "__main__":
    main()

```

---

### Solution

Only the score calculation in `forward` was missing:

```python
scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)
```

- `key.transpose(-2, -1)` swaps the last two dimensions of the key tensor, turning `(..., seq_len_k, d_k)` into `(..., d_k, seq_len_k)`. Using negative indices makes it work for both 3D (single-head) and 4D (multi-head) inputs.
- `torch.matmul(query, key.transpose(-2, -1))` computes the dot product between every query and every key, giving scores of shape `(batch_size, [num_heads,] seq_len_q, seq_len_k)`.
- Dividing by `math.sqrt(d_k)` applies the scaling factor $\frac{1}{\sqrt{d_k}}$, which keeps the variance of the scores roughly constant for any `d_k`, so softmax doesn't saturate and gradients stay healthy.
- Without this line, the later `scores.masked_fill(...)` and `F.softmax(scores, ...)` calls fail with `UnboundLocalError`, because `scores` is never defined.

Running the test shows the expected shapes: queries `(2, 5, 16)` and values `(2, 7, 20)` produce an output of `(2, 5, 20)` and attention weights of `(2, 5, 7)`, and the two masked key positions print `0.000000`.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class ScaledDotProductAttention(nn.Module):
    def __init__(self, dropout=0.1):
        """
        Scaled Dot-Product Attention module
        
        Args:
            dropout: Dropout rate
        """
        super(ScaledDotProductAttention, self).__init__()
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, query, key, value, mask=None):
        """
        Forward pass for scaled dot-product attention
        
        Args:
            query: (batch_size, num_heads, seq_len_q, d_k) or (batch_size, seq_len_q, d_k)
            key: (batch_size, num_heads, seq_len_k, d_k) or (batch_size, seq_len_k, d_k)
            value: (batch_size, num_heads, seq_len_v, d_v) or (batch_size, seq_len_v, d_v)
                   (seq_len_k and seq_len_v must be the same)
            mask: (batch_size, 1, seq_len_q, seq_len_k) or (batch_size, seq_len_q, seq_len_k)
                  Mask should be broadcastable.
                  
        Returns:
            output: (batch_size, num_heads, seq_len_q, d_v) or (batch_size, seq_len_q, d_v)
            attention_weights: (batch_size, num_heads, seq_len_q, seq_len_k) or (batch_size, seq_len_q, seq_len_k)
        """
        d_k = query.size(-1)
        
        # Compute attention scores with scaling: (B, [H], L_q, L_k)
        scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)
        
        # Apply mask if provided
        if mask is not None:
            # Ensure mask is broadcastable. Common mask shapes:
            # (B, 1, 1, L_k) for padding mask on keys
            # (B, 1, L_q, L_k) for combined padding/look-ahead
            # (1, 1, L_q, L_k) for look-ahead mask
            scores = scores.masked_fill(mask == 0, -1e9)
        
        # Apply softmax to get attention weights: (B, [H], L_q, L_k)
        attention_weights = F.softmax(scores, dim=-1)
        attention_weights = self.dropout(attention_weights) # Apply dropout to attention weights
        
        # Compute output by multiplying attention weights with values: (B, [H], L_q, d_v)
        output = torch.matmul(attention_weights, value)
        
        return output, attention_weights

def create_sample_qkv(batch_size, seq_len_q, seq_len_k, d_k, d_v):
    """Create sample Q, K, V tensors"""
    torch.manual_seed(42)
    q = torch.randn(batch_size, seq_len_q, d_k, requires_grad=True)
    k = torch.randn(batch_size, seq_len_k, d_k, requires_grad=True)
    v = torch.randn(batch_size, seq_len_k, d_v, requires_grad=True) # seq_len_k == seq_len_v
    return q, k, v
    
def main():
    print("Testing ScaledDotProductAttention Module")
    
    # Configuration
    batch_size = 2
    seq_len_q = 5
    seq_len_k = 7 # seq_len_v will be same as seq_len_k
    d_k = 16 # Dimension of keys/queries
    d_v = 20 # Dimension of values
    
    # Create sample data
    query, key, value = create_sample_qkv(batch_size, seq_len_q, seq_len_k, d_k, d_v)
    
    print(f"Input shapes - Q: {query.shape}, K: {key.shape}, V: {value.shape}")
    
    # Initialize attention module
    attention_module = ScaledDotProductAttention(dropout=0.1)
    
    # Test forward pass without mask
    output_no_mask, attn_weights_no_mask = attention_module(query, key, value)
    print(f"Output shape (no mask): {output_no_mask.shape}")
    print(f"Attention weights shape (no mask): {attn_weights_no_mask.shape}")
    assert output_no_mask.shape == (batch_size, seq_len_q, d_v)
    assert attn_weights_no_mask.shape == (batch_size, seq_len_q, seq_len_k)
    
    # Test with a mask
    mask = torch.ones(batch_size, seq_len_q, seq_len_k)
    mask[:, :, -2:] = 0 # Mask out last two key positions for all queries
    
    output_masked, attn_weights_masked = attention_module(query, key, value, mask=mask)
    print(f"Output shape (masked): {output_masked.shape}")
    print(f"Attention weights shape (masked): {attn_weights_masked.shape}")
    assert output_masked.shape == (batch_size, seq_len_q, d_v)
    
    # Verify masking worked: attention weights should be 0 for masked positions
    masked_positions = attn_weights_masked[:, :, -2:] # Last 2 key positions
    print(f"Masked positions attention (should be 0): {masked_positions.max():.6f}")
    

if __name__ == "__main__":
    main()


## Implementing Attention Mask Logic

Now that you've successfully computed the scaled attention scores, it's time to add masking support to make your attention module more robust. Masking is crucial in attention mechanisms because it allows you to ignore certain positions, like padding tokens or future tokens in language modeling.

---

### Your Task

In the provided code, you'll notice that the masking logic is incomplete. The `if mask is not None:` block contains a placeholder where you need to apply the mask to the attention scores. When a mask value is `0`, those positions should be set to a very small number (like `-1e9`) before the softmax operation, ensuring they receive near-zero attention weights.

Complete this masking step, and you'll have a fully functional attention module ready for real applications!

---

### Starter Code

```python
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class ScaledDotProductAttention(nn.Module):
    def __init__(self, dropout=0.1):
        """
        Scaled Dot-Product Attention module
        
        Args:
            dropout: Dropout rate
        """
        super(ScaledDotProductAttention, self).__init__()
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, query, key, value, mask=None):
        """
        Forward pass for scaled dot-product attention
        
        Args:
            query: (batch_size, num_heads, seq_len_q, d_k) or (batch_size, seq_len_q, d_k)
            key: (batch_size, num_heads, seq_len_k, d_k) or (batch_size, seq_len_k, d_k)
            value: (batch_size, num_heads, seq_len_v, d_v) or (batch_size, seq_len_v, d_v)
                   (seq_len_k and seq_len_v must be the same)
            mask: (batch_size, 1, seq_len_q, seq_len_k) or (batch_size, seq_len_q, seq_len_k)
                  Mask should be broadcastable.
                  
        Returns:
            output: (batch_size, num_heads, seq_len_q, d_v) or (batch_size, seq_len_q, d_v)
            attention_weights: (batch_size, num_heads, seq_len_q, seq_len_k) or (batch_size, seq_len_q, seq_len_k)
        """
        d_k = query.size(-1)
        
        # Compute attention scores with scaling: (B, [H], L_q, L_k)
        scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)
        
        # Apply mask if provided
        if mask is not None:
            # TODO: Apply the mask to the scores
            pass
        
        # Apply softmax to get attention weights: (B, [H], L_q, L_k)
        attention_weights = F.softmax(scores, dim=-1)
        attention_weights = self.dropout(attention_weights) # Apply dropout to attention weights
        
        # Compute output by multiplying attention weights with values: (B, [H], L_q, d_v)
        output = torch.matmul(attention_weights, value)
        
        return output, attention_weights

def create_sample_qkv(batch_size, seq_len_q, seq_len_k, d_k, d_v):
    """Create sample Q, K, V tensors"""
    torch.manual_seed(42)
    q = torch.randn(batch_size, seq_len_q, d_k, requires_grad=True)
    k = torch.randn(batch_size, seq_len_k, d_k, requires_grad=True)
    v = torch.randn(batch_size, seq_len_k, d_v, requires_grad=True) # seq_len_k == seq_len_v
    return q, k, v
    
def main():
    print("Testing ScaledDotProductAttention Module")
    
    # Configuration
    batch_size = 2
    seq_len_q = 5
    seq_len_k = 7 # seq_len_v will be same as seq_len_k
    d_k = 16 # Dimension of keys/queries
    d_v = 20 # Dimension of values
    
    # Create sample data
    query, key, value = create_sample_qkv(batch_size, seq_len_q, seq_len_k, d_k, d_v)
    
    print(f"Input shapes - Q: {query.shape}, K: {key.shape}, V: {value.shape}")
    
    # Initialize attention module
    attention_module = ScaledDotProductAttention(dropout=0.1)
    
    # Test forward pass without mask
    output_no_mask, attn_weights_no_mask = attention_module(query, key, value)
    print(f"Output shape (no mask): {output_no_mask.shape}")
    print(f"Attention weights shape (no mask): {attn_weights_no_mask.shape}")
    assert output_no_mask.shape == (batch_size, seq_len_q, d_v)
    assert attn_weights_no_mask.shape == (batch_size, seq_len_q, seq_len_k)
    
    # Test with a mask
    mask = torch.ones(batch_size, seq_len_q, seq_len_k)
    mask[:, :, -2:] = 0 # Mask out last two key positions for all queries
    
    output_masked, attn_weights_masked = attention_module(query, key, value, mask=mask)
    print(f"Output shape (masked): {output_masked.shape}")
    print(f"Attention weights shape (masked): {attn_weights_masked.shape}")
    assert output_masked.shape == (batch_size, seq_len_q, d_v)
    
    # Verify masking worked: attention weights should be 0 for masked positions
    masked_positions = attn_weights_masked[:, :, -2:] # Last 2 key positions
    print(f"Masked positions attention (should be 0): {masked_positions.max():.6f}")
    

if __name__ == "__main__":
    main()

```

---

### Solution

Only the mask application inside `forward` was missing:

```python
if mask is not None:
    scores = scores.masked_fill(mask == 0, -1e9)
```

- `mask == 0` selects every position marked as blocked: padding tokens, future tokens, or any other position that should be ignored.
- `masked_fill(..., -1e9)` replaces those scores with a very large negative number, so after softmax $e^{-10^9} \approx 0$ and the blocked positions receive essentially zero attention weight.
- Allowed positions keep their original scores, and softmax redistributes all of the weight among them, so each row of attention weights still sums to 1.
- The mask only needs to be broadcastable to the score shape `(batch_size, [num_heads,] seq_len_q, seq_len_k)`, so padding masks, look-ahead masks and combined masks all work with the same line.
- Without the fix, the `pass` placeholder leaves the scores untouched, so masked positions still receive attention.

In the test, the last two key positions are masked for every query, and the final check now prints `0.000000` for them, confirming they receive no attention.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class ScaledDotProductAttention(nn.Module):
    def __init__(self, dropout=0.1):
        """
        Scaled Dot-Product Attention module
        
        Args:
            dropout: Dropout rate
        """
        super(ScaledDotProductAttention, self).__init__()
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, query, key, value, mask=None):
        """
        Forward pass for scaled dot-product attention
        
        Args:
            query: (batch_size, num_heads, seq_len_q, d_k) or (batch_size, seq_len_q, d_k)
            key: (batch_size, num_heads, seq_len_k, d_k) or (batch_size, seq_len_k, d_k)
            value: (batch_size, num_heads, seq_len_v, d_v) or (batch_size, seq_len_v, d_v)
                   (seq_len_k and seq_len_v must be the same)
            mask: (batch_size, 1, seq_len_q, seq_len_k) or (batch_size, seq_len_q, seq_len_k)
                  Mask should be broadcastable.
                  
        Returns:
            output: (batch_size, num_heads, seq_len_q, d_v) or (batch_size, seq_len_q, d_v)
            attention_weights: (batch_size, num_heads, seq_len_q, seq_len_k) or (batch_size, seq_len_q, seq_len_k)
        """
        d_k = query.size(-1)
        
        # Compute attention scores with scaling: (B, [H], L_q, L_k)
        scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)
        
        # Apply mask if provided
        if mask is not None:
            # Blocked positions (mask == 0) get a very negative score, so softmax makes them ~0
            scores = scores.masked_fill(mask == 0, -1e9)
        
        # Apply softmax to get attention weights: (B, [H], L_q, L_k)
        attention_weights = F.softmax(scores, dim=-1)
        attention_weights = self.dropout(attention_weights) # Apply dropout to attention weights
        
        # Compute output by multiplying attention weights with values: (B, [H], L_q, d_v)
        output = torch.matmul(attention_weights, value)
        
        return output, attention_weights

def create_sample_qkv(batch_size, seq_len_q, seq_len_k, d_k, d_v):
    """Create sample Q, K, V tensors"""
    torch.manual_seed(42)
    q = torch.randn(batch_size, seq_len_q, d_k, requires_grad=True)
    k = torch.randn(batch_size, seq_len_k, d_k, requires_grad=True)
    v = torch.randn(batch_size, seq_len_k, d_v, requires_grad=True) # seq_len_k == seq_len_v
    return q, k, v
    
def main():
    print("Testing ScaledDotProductAttention Module")
    
    # Configuration
    batch_size = 2
    seq_len_q = 5
    seq_len_k = 7 # seq_len_v will be same as seq_len_k
    d_k = 16 # Dimension of keys/queries
    d_v = 20 # Dimension of values
    
    # Create sample data
    query, key, value = create_sample_qkv(batch_size, seq_len_q, seq_len_k, d_k, d_v)
    
    print(f"Input shapes - Q: {query.shape}, K: {key.shape}, V: {value.shape}")
    
    # Initialize attention module
    attention_module = ScaledDotProductAttention(dropout=0.1)
    
    # Test forward pass without mask
    output_no_mask, attn_weights_no_mask = attention_module(query, key, value)
    print(f"Output shape (no mask): {output_no_mask.shape}")
    print(f"Attention weights shape (no mask): {attn_weights_no_mask.shape}")
    assert output_no_mask.shape == (batch_size, seq_len_q, d_v)
    assert attn_weights_no_mask.shape == (batch_size, seq_len_q, seq_len_k)
    
    # Test with a mask
    mask = torch.ones(batch_size, seq_len_q, seq_len_k)
    mask[:, :, -2:] = 0 # Mask out last two key positions for all queries
    
    output_masked, attn_weights_masked = attention_module(query, key, value, mask=mask)
    print(f"Output shape (masked): {output_masked.shape}")
    print(f"Attention weights shape (masked): {attn_weights_masked.shape}")
    assert output_masked.shape == (batch_size, seq_len_q, d_v)
    
    # Verify masking worked: attention weights should be 0 for masked positions
    masked_positions = attn_weights_masked[:, :, -2:] # Last 2 key positions
    print(f"Masked positions attention (should be 0): {masked_positions.max():.6f}")
    

if __name__ == "__main__":
    main()


## Complete the Attention Pipeline